# Can we predict `passed_quiz`?
Protocol: `protocol.yaml` (group holdout on `student_key` via `holdout.csv`, 5-fold GroupKFold CV on training rows, choose by mean CV ROC AUC, score holdout once).

**Feature timing (from `columns.csv`).** We predict before the closing quiz, i.e. at session start. `session_minutes` is the length of the whole session, which ends with the quiz, so it is only known afterwards; `hints_used` is accumulated during the session. Both are excluded. Used: `enrol_year`, `school_code` (treated as a categorical code), `prior_score` (placement test, earlier than the session). `student_key` is excluded per protocol.

In [ ]:
import json, hashlib
import numpy as np, pandas as pd
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score

df = pd.read_csv('data.csv')
assert hashlib.sha256(open('data.csv','rb').read()).hexdigest()=='5a0b0221ed641db40322e1a95cae32cc67d3525ecb500813f95d6c109b2a7409'
hold_rows = pd.read_csv('holdout.csv')['row'].to_numpy()
assert len(df)==12453 and len(hold_rows)==2465
train_mask = ~df.index.isin(hold_rows)
train, hold = df[train_mask], df[~train_mask]
assert not set(train.student_key) & set(hold.student_key)
FEATURES = ['enrol_year','school_code','prior_score']
TARGET = 'passed_quiz'
print(len(train), len(hold), train[TARGET].mean(), hold[TARGET].mean())

## Candidates (all named before any scoring)

In [ ]:
num = ['enrol_year','prior_score']
def pre(onehot_school):
    if onehot_school:
        return ColumnTransformer([('n',StandardScaler(),num),('c',OneHotEncoder(handle_unknown='ignore',min_frequency=20),['school_code'])])
    return ColumnTransformer([('n',StandardScaler(),num)])
candidates = {
 'baseline_prior': lambda: DummyClassifier(strategy='prior'),
 'logreg_C1': lambda: make_pipeline(pre(True), LogisticRegression(C=1.0, max_iter=1000)),
 'logreg_C0.1': lambda: make_pipeline(pre(True), LogisticRegression(C=0.1, max_iter=1000)),
 'logreg_noschool': lambda: make_pipeline(pre(False), LogisticRegression(C=1.0, max_iter=1000)),
 'hgb_depth3': lambda: HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=200, random_state=0),
}
print(list(candidates))

## Cross-validation on training rows (GroupKFold by student)

In [ ]:
gkf = GroupKFold(n_splits=5)
folds = list(gkf.split(train[FEATURES], train[TARGET], groups=train['student_key']))
cv_scores, cv_folds = {}, []
for name, mk in candidates.items():
    s=[]
    for tr, va in folds:
        m = mk().fit(train.iloc[tr][FEATURES], train.iloc[tr][TARGET])
        s.append(roc_auc_score(train.iloc[va][TARGET], m.predict_proba(train.iloc[va][FEATURES])[:,1]))
    cv_scores[name] = s
for tr, va in folds:
    cv_folds.append([[int(i) for i in train.index[tr]], [int(i) for i in train.index[va]]])
summary = pd.DataFrame({k:{'mean':np.mean(v),'std':np.std(v)} for k,v in cv_scores.items()}).T
summary

## Choose by mean CV ROC AUC, refit on all training rows, score holdout once

In [ ]:
chosen = max(cv_scores, key=lambda k: np.mean(cv_scores[k]))
print('chosen:', chosen)
final = candidates[chosen]().fit(train[FEATURES], train[TARGET])
holdout_auc = roc_auc_score(hold[TARGET], final.predict_proba(hold[FEATURES])[:,1])
print('holdout roc_auc:', holdout_auc)

In [ ]:
evidence = {
 'features': FEATURES,
 'candidates': list(candidates),
 'cv_folds': cv_folds,
 'cv_scores': {k:[float(x) for x in v] for k,v in cv_scores.items()},
 'chosen': chosen,
 'holdout_rows': [int(i) for i in hold.index],
 'holdout_scores': {chosen: float(holdout_auc)},
}
json.dump(evidence, open('evidence.json','w'))

## Conclusion
See the CV table and holdout ROC AUC above. Only pre-session features were used; in-session variables (`session_minutes`, `hints_used`) would answer a different, after-the-fact question.